In [4]:
from dynamicTasteDistortion.dataset_loader import read_yelp_raw
from dynamicTasteDistortion.scripts.data_utils import preprocess_genres

# Yelp dataset EDA

In [2]:
yelp = read_yelp_raw(1_000_0000)

Criando o diretorio dynamicTasteDistortion/data/yelp/raw/yelp_10000000/
Downloading...
Unzipping...


In [3]:
yelp

,user,item,rating,date,genres
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B..."
1,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,2012-01-03 15:28:18,"Active Life, Cycling Classes, Trainers, Gyms, ..."
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Restaurants, Breakfast & Brunch"
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,2015-01-04 00:01:03,"Halal, Pakistani, Restaurants, Indian"
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,2017-01-14 20:54:15,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ..."
...,...,...,...,...,...
6990275,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,2014-12-17 21:45:20,"Home Services, Mortgage Brokers, Real Estate, ..."
6990276,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2021-03-31 16:55:10,"Hotels & Travel, Active Life, Boating, Rafting..."
6990277,mm6E4FbCMwJmb7kPDZ5v2Q,R1khUUxidqfaJmcpmGd4aw,4.0,2019-12-30 03:56:30,"Building Supplies, Home Services, Hardware Sto..."
6990278,YwAMC-jvZ1fvEUum6QkEkw,Rr9kKArrMhSLVE9a53q-aA,5.0,2022-01-19 18:59:27,"Desserts, Pizza, Salad, Food, Beer, Wine & Spi..."


### Number of interactions per genre


**TL;DR**: Filter out genres that are bellow the 50% percentile. Because items are multi-category, this filter reduces around 400k interactions from the raw dataset

In [10]:
yelp

,user,item,rating,date,genres
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B..."
1,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,2012-01-03 15:28:18,"Active Life, Cycling Classes, Trainers, Gyms, ..."
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Restaurants, Breakfast & Brunch"
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,2015-01-04 00:01:03,"Halal, Pakistani, Restaurants, Indian"
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,2017-01-14 20:54:15,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ..."
...,...,...,...,...,...
6990275,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,2014-12-17 21:45:20,"Home Services, Mortgage Brokers, Real Estate, ..."
6990276,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2021-03-31 16:55:10,"Hotels & Travel, Active Life, Boating, Rafting..."
6990277,mm6E4FbCMwJmb7kPDZ5v2Q,R1khUUxidqfaJmcpmGd4aw,4.0,2019-12-30 03:56:30,"Building Supplies, Home Services, Hardware Sto..."
6990278,YwAMC-jvZ1fvEUum6QkEkw,Rr9kKArrMhSLVE9a53q-aA,5.0,2022-01-19 18:59:27,"Desserts, Pizza, Salad, Food, Beer, Wine & Spi..."


In [11]:
yelp["genres"]

0          Restaurants, Breakfast & Brunch, Food, Juice B...
1          Active Life, Cycling Classes, Trainers, Gyms, ...
2                            Restaurants, Breakfast & Brunch
3                      Halal, Pakistani, Restaurants, Indian
4          Sandwiches, Beer, Wine & Spirits, Bars, Food, ...
                                 ...                        
6990275    Home Services, Mortgage Brokers, Real Estate, ...
6990276    Hotels & Travel, Active Life, Boating, Rafting...
6990277    Building Supplies, Home Services, Hardware Sto...
6990278    Desserts, Pizza, Salad, Food, Beer, Wine & Spi...
6990279    Burgers, Sandwiches, Food, Beer, Wine & Spirit...
Name: genres, Length: 6990280, dtype: object

In [15]:
yelp["genres"].iloc[0].split(",")

['Restaurants',
 ' Breakfast & Brunch',
 ' Food',
 ' Juice Bars & Smoothies',
 ' American (New)',
 ' Coffee & Tea',
 ' Sandwiches']

In [19]:
yelp[yelp["genres"].isna()]

,user,item,rating,date,genres
2274,NMnvdoNDYN92DlXJJtDFRw,ZERQMWb1PFzCfbfknqq-fA,1.0,2017-12-15 23:21:01,None
53849,l1TVWV45iN6QVzw--oW8qA,9QoKKDZB_YuDeS5TxRW8bg,1.0,2017-02-11 21:19:36,None
58878,P4xaN82k39NPSC8Qxh40mg,9QoKKDZB_YuDeS5TxRW8bg,1.0,2016-12-05 20:29:27,None
83402,DRTVAPqhMpmXrNJbT-dy7Q,mKxCNYEoKt6d_1rXmvRwww,4.0,2017-05-02 22:06:31,None
142278,FGgnfpTySh4OKrKvhEWcYg,9QoKKDZB_YuDeS5TxRW8bg,1.0,2016-02-09 21:35:24,None
...,...,...,...,...,...
6970857,fK-FaH7m_C3JPKrWPAqbKQ,AejQfc55_PtIF-SIFIYGHg,5.0,2019-06-28 22:31:34,None
6975272,jc6QygrpVrHIWASWcwe1cA,jAY0Or7fZa1lR4xSUEEEpA,1.0,2018-07-13 21:56:57,None
6976774,jL_Ux88q9L4ZP_iSbxkHuQ,DCvA43gLeetay_qttR9ABQ,1.0,2012-06-01 20:37:08,None
6986978,1_JeDgLoAcyBL87FAT4xNA,AejQfc55_PtIF-SIFIYGHg,5.0,2022-01-05 02:23:03,None


In [59]:
yelp_cleaned = yelp[~yelp["genres"].isna()]

In [21]:
yelp_cleaned

,user,item,rating,date,genres
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B..."
1,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,2012-01-03 15:28:18,"Active Life, Cycling Classes, Trainers, Gyms, ..."
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Restaurants, Breakfast & Brunch"
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,2015-01-04 00:01:03,"Halal, Pakistani, Restaurants, Indian"
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,2017-01-14 20:54:15,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ..."
...,...,...,...,...,...
6990275,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,2014-12-17 21:45:20,"Home Services, Mortgage Brokers, Real Estate, ..."
6990276,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2021-03-31 16:55:10,"Hotels & Travel, Active Life, Boating, Rafting..."
6990277,mm6E4FbCMwJmb7kPDZ5v2Q,R1khUUxidqfaJmcpmGd4aw,4.0,2019-12-30 03:56:30,"Building Supplies, Home Services, Hardware Sto..."
6990278,YwAMC-jvZ1fvEUum6QkEkw,Rr9kKArrMhSLVE9a53q-aA,5.0,2022-01-19 18:59:27,"Desserts, Pizza, Salad, Food, Beer, Wine & Spi..."


In [23]:
yelp_cleaned["genres_processed"] = preprocess_genres(yelp_cleaned, "genres", SEP=",")

In [28]:
full_blown_interaction_yelp = yelp_cleaned.explode(["genres_processed"])

In [39]:
interactions_per_genre = full_blown_interaction_yelp.groupby("genres_processed").size()
quantile_50p = interactions_per_genre.sort_values().quantile(0.5)

In [41]:
interactions_per_genre

genres_processed
&-probates       329
3d-printing       71
acai-bowls     16621
accessories    28610
accountants     1184
               ...  
wraps          19422
yelp-events     1705
yoga           20190
ziplining        728
zoos            7926
Length: 1311, dtype: int64

In [43]:
    interactions_per_genre > 330

genres_processed
&-probates     False
3d-printing    False
acai-bowls      True
accessories     True
accountants     True
               ...  
wraps           True
yelp-events     True
yoga            True
ziplining       True
zoos            True
Length: 1311, dtype: bool

In [52]:
genres_to_filter_out = list(interactions_per_genre[interactions_per_genre < quantile_50p].index)

In [53]:
genres_to_filter_out

['&-probates',
 '3d-printing',
 'accountants',
 'acne-treatment',
 'addiction-medicine',
 'adoption-services',
 'advertising',
 'aerial-fitness',
 'aerial-tours',
 'aestheticians',
 'aircraft-dealers',
 'aircraft-repairs',
 'airport-lounges',
 'airport-terminals',
 'airsoft',
 'allergists',
 'alternative-medicine',
 'anesthesiologists',
 'animal-assisted-therapy',
 'animal-physical-therapy',
 'apartment-agents',
 'appraisal-services',
 'aquarium-services',
 'archery',
 'architects',
 'architectural-tours',
 'art-consultants',
 'art-installation',
 'art-restoration',
 'art-space-rentals',
 'artificial-turf',
 'assisted-living-facilities',
 'astrologers',
 'attraction-farms',
 'atv-rentals/tours',
 'auction-houses',
 'audio/visual-equipment-rental',
 'audiologist',
 'australian',
 'austrian',
 'auto-security',
 'auto-upholstery',
 'aviation-services',
 'awnings',
 'ayurveda',
 'backflow-services',
 'badminton',
 'bail-bondsmen',
 'balloon-services',
 'bangladeshi',
 'bankruptcy-law',
 'b

In [58]:
"pakistani" in genres_to_filter_out

False

In [55]:
full_blown_interaction_yelp

,user,item,rating,date,genres,genres_processed
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...",restaurants
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...",breakfast-&-brunch
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...",food
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...",juice-bars-&-smoothies
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...",american-(new)
...,...,...,...,...,...,...
6990279,6JehEvdoCvZPJ_XIxnzIIw,VAeEXLbEcI9Emt9KGYq9aA,3.0,2018-01-02 22:50:47,"Burgers, Sandwiches, Food, Beer, Wine & Spirit...",wine-&-spirits
6990279,6JehEvdoCvZPJ_XIxnzIIw,VAeEXLbEcI9Emt9KGYq9aA,3.0,2018-01-02 22:50:47,"Burgers, Sandwiches, Food, Beer, Wine & Spirit...",american-(traditional)
6990279,6JehEvdoCvZPJ_XIxnzIIw,VAeEXLbEcI9Emt9KGYq9aA,3.0,2018-01-02 22:50:47,"Burgers, Sandwiches, Food, Beer, Wine & Spirit...",bars
6990279,6JehEvdoCvZPJ_XIxnzIIw,VAeEXLbEcI9Emt9KGYq9aA,3.0,2018-01-02 22:50:47,"Burgers, Sandwiches, Food, Beer, Wine & Spirit...",nightlife


In [71]:
filtered_full_blown = full_blown_interaction_yelp[
    ~full_blown_interaction_yelp["genres_processed"].isin(genres_to_filter_out)
]

In [ ]:
before = full_blown_interaction_yelp.copy()

In [69]:
mask = before["genres_processed"].apply(lambda r: "courthouses" in r)
before[mask == True]

,user,item,rating,date,genres,genres_processed
88,9QpNZ4-0gJow6xj-6OZe8Q,6JFTijOMHB46yBoyVOjPCA,4.0,2015-08-30 16:48:31,"Courthouses, Active Life, Event Planning & Ser...",courthouses
4665,AIBCHP4azbrP_ASpcIjSXA,6JFTijOMHB46yBoyVOjPCA,5.0,2016-01-18 00:15:33,"Courthouses, Active Life, Event Planning & Ser...",courthouses
4729,a40SmVr_vopXYhtcyupBAw,6JFTijOMHB46yBoyVOjPCA,4.0,2015-10-08 07:12:40,"Courthouses, Active Life, Event Planning & Ser...",courthouses
6641,lQ3xOu_L76UUST8tPPaWDQ,6JFTijOMHB46yBoyVOjPCA,5.0,2017-02-19 17:00:22,"Courthouses, Active Life, Event Planning & Ser...",courthouses
7104,mdEQeq2PHG4VqX4LigLLig,6JFTijOMHB46yBoyVOjPCA,5.0,2008-04-28 22:23:26,"Courthouses, Active Life, Event Planning & Ser...",courthouses
...,...,...,...,...,...,...
6601108,vqzWERrURYgK1-kmHheaxA,z99C9uYanRWZpTbyQ6yHnA,1.0,2018-10-08 23:16:16,"Public Services & Government, Courthouses",courthouses
6751474,DB5uuOamJ5N2RZ5ylqOFQw,z99C9uYanRWZpTbyQ6yHnA,3.0,2019-05-31 03:40:33,"Public Services & Government, Courthouses",courthouses
6766697,YELlwStpNGsV9pUibxCbdQ,z99C9uYanRWZpTbyQ6yHnA,1.0,2016-06-30 22:41:04,"Public Services & Government, Courthouses",courthouses
6836722,djkK-WG1xt7BBUnhxNOHYw,z99C9uYanRWZpTbyQ6yHnA,1.0,2012-08-18 03:32:23,"Public Services & Government, Courthouses",courthouses


In [72]:
mask = filtered_full_blown["genres_processed"].apply(lambda r: "courthouses" in r)
filtered_full_blown[mask == True]

,user,item,rating,date,genres,genres_processed


In [73]:
yelp_cleaned['genres_processed'] = filtered_full_blown.groupby(level=0)['genres_processed'].agg(list)
yelp_cleaned

,user,item,rating,date,genres,genres_processed
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...","[restaurants, breakfast-&-brunch, food, juice-..."
1,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,2012-01-03 15:28:18,"Active Life, Cycling Classes, Trainers, Gyms, ...","[active-life, cycling-classes, trainers, gyms,..."
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Restaurants, Breakfast & Brunch","[restaurants, breakfast-&-brunch]"
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,2015-01-04 00:01:03,"Halal, Pakistani, Restaurants, Indian","[halal, pakistani, restaurants, indian]"
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,2017-01-14 20:54:15,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ...","[sandwiches, beer, wine-&-spirits, bars, food,..."
...,...,...,...,...,...,...
6990275,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,2014-12-17 21:45:20,"Home Services, Mortgage Brokers, Real Estate, ...","[home-services, mortgage-brokers, real-estate,..."
6990276,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2021-03-31 16:55:10,"Hotels & Travel, Active Life, Boating, Rafting...","[hotels-&-travel, active-life, boating, raftin..."
6990277,mm6E4FbCMwJmb7kPDZ5v2Q,R1khUUxidqfaJmcpmGd4aw,4.0,2019-12-30 03:56:30,"Building Supplies, Home Services, Hardware Sto...","[building-supplies, home-services, hardware-st..."
6990278,YwAMC-jvZ1fvEUum6QkEkw,Rr9kKArrMhSLVE9a53q-aA,5.0,2022-01-19 18:59:27,"Desserts, Pizza, Salad, Food, Beer, Wine & Spi...","[desserts, pizza, salad, food, beer, wine-&-sp..."
